# Real-event inference sanity check: M08 + Mondrian conformal prediction

This notebook explores whether the M08 CNN + Mondrian conformal prediction pipeline trained on simulated BBH signals can be applied to real GWOSC/LVK events as a first-order low-latency parameter-estimation method.

The goal is not to claim formal conformal validity on real detector data. The conformal calibration was performed on simulated signals, so real-event inference is affected by domain shift: real non-stationary detector noise, glitches, calibration uncertainty, PSD mismatch, detector availability, and waveform-systematics differences.

The notebook proceeds incrementally:

1. Load the exact training/generation configuration.
2. Reconstruct the M08 model.
3. Load label normalization statistics.
4. Validate inference on controlled inputs.
5. Reconstruct/apply Mondrian calibrators.
6. Build real GWOSC event inputs.
7. Compare predictions and intervals with published LVK values.

Construir el input real para uno o pocos eventos HLV, pasar M08, aplicar el calibrador Mondrian y comparar cualitativamente con LVK. No estimar cobertura real todavía.

## 1. Imports and Paths

In [32]:
from pathlib import Path
import json
import os
import sys
import numpy as np
import torch
import matplotlib.pyplot as plt

# Ajusta estas rutas a tu entorno actual

print("Current working directory:", os.getcwd())

PROJECT_ROOT = Path("/home/victor/gw/cbc_pe")
DATA_ROOT = Path("/home/victor/gw/cbc_pe/data")
#PROJECT_ROOT = Path("/data/vserrano/gw/Gravitational-Waves-Lab/cbc_pe")
#DATA_ROOT = Path("/data/vserrano/cbc_pe/data")

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f"PROJECT_ROOT does not exist: {PROJECT_ROOT}")
else:
    sys.path.insert(0, str(PROJECT_ROOT))

DATASET_ID = "bbh_processed_4s_seobnrv4opt_snr10-25_n500_000"

CONFIG_DIR = PROJECT_ROOT / "configs" 
MODEL_DIR = DATA_ROOT / "models" / "checkpoints" / DATASET_ID
RESULTS_DIR = DATA_ROOT / "results" / DATASET_ID

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("CONFIG_DIR exists:", CONFIG_DIR.exists())
print("MODEL_DIR exists:", MODEL_DIR.exists())
print("RESULTS_DIR exists:", RESULTS_DIR.exists())

Current working directory: /home/victor/gw/cbc_pe/notebooks
PROJECT_ROOT: /home/victor/gw/cbc_pe
DATA_ROOT: /home/victor/gw/cbc_pe/data
CONFIG_DIR exists: True
MODEL_DIR exists: True
RESULTS_DIR exists: True


## 2. Load configs and parameters

In [33]:
generation_config_path =  PROJECT_ROOT / "configs" / "generation" / "generate_500k_bbh_4s.json"
training_config_path = PROJECT_ROOT / "configs" / "experiments" / "train_500k_M08_resdilated_emb64_d124_bs256_seed123.json"

with open(generation_config_path, "r") as f:
    gen_cfg = json.load(f)

with open(training_config_path, "r") as f:
    train_cfg = json.load(f)

print("Generation config keys:", gen_cfg.keys())
print("Training config keys:", train_cfg.keys())
print("Training parameters:", train_cfg["model"]["kwargs"].keys())

Generation config keys: dict_keys(['project_root', 'data_root', 'output', 'generation', 'simulation', 'parameter_sampler', 'detectors', 'signal_processor', 'label_transformer'])
Training config keys: dict_keys(['project_root', 'data_root', 'dataset', 'model', 'training', 'outputs'])
Training parameters: dict_keys(['n_detectors', 'n_outputs', 'embedding_dim', 'residual_channels', 'dilations', 'residual_kernel_size', 'dropout_conv', 'dropout_dense', 'num_groups'])


## 3. Signal requirements / conditions

In [34]:
detectors = gen_cfg["detectors"]

fs = 4096
duration = gen_cfg["simulation"]["duration"]
n_samples = int(duration * fs)

context_start = gen_cfg["simulation"]["processing_context_start_samples"]
context_end = gen_cfg["simulation"]["processing_context_end_samples"]
processing_length = n_samples + context_start + context_end

signal_processor_cfg = gen_cfg["signal_processor"]

print("Detector order:", detectors)
print("Sampling frequency:", fs)
print("Final duration:", duration)
print("Final samples:", n_samples)
print("Processing context start samples:", context_start)
print("Processing context end samples:", context_end)
print("Processing input length:", processing_length)
print("Processing input duration:", processing_length / fs)

print("\nSignal processor:")
for k, v in signal_processor_cfg.items():
    print(f"  {k}: {v}")

Detector order: ['H1', 'L1', 'V1']
Sampling frequency: 4096
Final duration: 4.0
Final samples: 16384
Processing context start samples: 1664
Processing context end samples: 1664
Processing input length: 19712
Processing input duration: 4.8125

Signal processor:
  whitening_method: psd
  apply_highpass: True
  apply_lowpass: True
  apply_standardization: False
  output_mode: crop_to_config
  whitening_low_frequency_cutoff: 30.0
  whitening_max_filter_duration: 0.5
  whitening_trunc_method: hann
  highpass_frequency: 30.0
  lowpass_frequency: 512.0
  fir_order: 256
  fir_beta: 5.0
  remove_corrupted: True


In [35]:
### SANITY CHECKS to assure that the configuration parameters are consistent with the expected values

assert detectors == ["H1", "L1", "V1"], detectors
assert fs == 4096
assert n_samples == 16384
assert processing_length == 19712

assert signal_processor_cfg["whitening_method"] == "psd"
assert signal_processor_cfg["apply_highpass"] is True
assert signal_processor_cfg["apply_lowpass"] is True
assert signal_processor_cfg["apply_standardization"] is False
assert signal_processor_cfg["highpass_frequency"] == 30.0
assert signal_processor_cfg["lowpass_frequency"] == 512.0

print("Input contract validated.")

Input contract validated.


## 4. Imports (model, simulation-config)

In [36]:
from src.models.network import SimpleCNN_ResidualDilated

# Getting model info
model_cfg = train_cfg["model"]

print(model_cfg["class_name"])
print(model_cfg["kwargs"])

model = SimpleCNN_ResidualDilated(**model_cfg["kwargs"])
model.eval()

n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(model)
print(f"Total parameters: {n_params:,}")
print(f"Trainable parameters: {n_trainable:,}")

SimpleCNN_ResidualDilated
{'n_detectors': 3, 'n_outputs': 3, 'embedding_dim': 64, 'residual_channels': 64, 'dilations': [1, 2, 4], 'residual_kernel_size': 7, 'dropout_conv': 0.05, 'dropout_dense': 0.1, 'num_groups': 8}
SimpleCNN_ResidualDilated(
  (block1): ConvBlock(
    (conv): Conv1d(3, 16, kernel_size=(16,), stride=(2,), padding=(8,))
    (group_norm): GroupNorm(8, 16, eps=1e-05, affine=True)
    (activation): LeakyReLU(negative_slope=0.01)
    (dropout): Dropout(p=0.05, inplace=False)
  )
  (block2): ConvBlock(
    (conv): Conv1d(16, 32, kernel_size=(16,), stride=(2,), padding=(8,))
    (group_norm): GroupNorm(8, 32, eps=1e-05, affine=True)
    (activation): LeakyReLU(negative_slope=0.01)
    (dropout): Dropout(p=0.05, inplace=False)
  )
  (block3): ConvBlock(
    (conv): Conv1d(32, 64, kernel_size=(16,), stride=(2,), padding=(8,))
    (group_norm): GroupNorm(8, 64, eps=1e-05, affine=True)
    (activation): LeakyReLU(negative_slope=0.01)
    (dropout): Dropout(p=0.05, inplace=Fals

In [37]:
### Shape test

x_dummy = torch.zeros((2, 3, 16384), dtype=torch.float32)

with torch.no_grad():
    y_dummy, emb_dummy = model(x_dummy, return_embedding=True)

print("y_dummy shape:", y_dummy.shape)
print("emb_dummy shape:", emb_dummy.shape)

y_dummy shape: torch.Size([2, 3])
emb_dummy shape: torch.Size([2, 64])


## 5. Load the checkpoint

In [38]:
!ls  /home/victor/gw/cbc_pe/data/models/checkpoints/bbh_processed_4s_seobnrv4opt_snr10-25_n500_000

bbh_processed_4s_seobnrv4opt_snr10-25_n500_000_SimpleCNN_ResidualDilated_M08_resdilated_emb64_d124_seed123_checkpoint.pt


In [39]:
print (MODEL_DIR)

/home/victor/gw/cbc_pe/data/models/checkpoints/bbh_processed_4s_seobnrv4opt_snr10-25_n500_000


In [40]:
checkpoint_tag = train_cfg["outputs"]["checkpoint_tag"]
print("Checkpoint tag:", checkpoint_tag)

candidate_checkpoints = sorted(MODEL_DIR.rglob(f"*{checkpoint_tag}*"))
for p in candidate_checkpoints[:20]:
    print(p)

print("Number of candidates:", len(candidate_checkpoints))

Checkpoint tag: M08_resdilated_emb64_d124
/home/victor/gw/cbc_pe/data/models/checkpoints/bbh_processed_4s_seobnrv4opt_snr10-25_n500_000/bbh_processed_4s_seobnrv4opt_snr10-25_n500_000_SimpleCNN_ResidualDilated_M08_resdilated_emb64_d124_seed123_checkpoint.pt
Number of candidates: 1


In [41]:
checkpoint_path = candidate_checkpoints[-1]  # Load the last checkpoint

ckpt = torch.load(checkpoint_path, map_location="cpu")
print(ckpt.keys())

model.load_state_dict(ckpt["model_state_dict"])
model.eval()

dict_keys(['epoch', 'model_state_dict', 'optimizer_state_dict', 'train_loss', 'best_val_loss', 'y_mean', 'y_std', 'model_config', 'training_config', 'elapsed_seconds', 'history'])


SimpleCNN_ResidualDilated(
  (block1): ConvBlock(
    (conv): Conv1d(3, 16, kernel_size=(16,), stride=(2,), padding=(8,))
    (group_norm): GroupNorm(8, 16, eps=1e-05, affine=True)
    (activation): LeakyReLU(negative_slope=0.01)
    (dropout): Dropout(p=0.05, inplace=False)
  )
  (block2): ConvBlock(
    (conv): Conv1d(16, 32, kernel_size=(16,), stride=(2,), padding=(8,))
    (group_norm): GroupNorm(8, 32, eps=1e-05, affine=True)
    (activation): LeakyReLU(negative_slope=0.01)
    (dropout): Dropout(p=0.05, inplace=False)
  )
  (block3): ConvBlock(
    (conv): Conv1d(32, 64, kernel_size=(16,), stride=(2,), padding=(8,))
    (group_norm): GroupNorm(8, 64, eps=1e-05, affine=True)
    (activation): LeakyReLU(negative_slope=0.01)
    (dropout): Dropout(p=0.05, inplace=False)
  )
  (residual_blocks): Sequential(
    (0): ResidualDilatedBlock(
      (conv1): Conv1d(64, 64, kernel_size=(7,), stride=(1,), padding=(3,))
      (norm1): GroupNorm(8, 64, eps=1e-05, affine=True)
      (activation

In [42]:
with torch.no_grad():
    y_dummy, emb_dummy = model(x_dummy, return_embedding=True)

print(y_dummy)
print(emb_dummy.shape)

tensor([[-2.0863, -2.5918, -0.5304],
        [-2.0863, -2.5918, -0.5304]])
torch.Size([2, 64])


## 6. Label normalization statistics

In [43]:
# ------------------------------------------------------------
# 6. Label normalization statistics
# ------------------------------------------------------------

y_mean = np.asarray(ckpt["y_mean"], dtype=np.float64)
y_std = np.asarray(ckpt["y_std"], dtype=np.float64)

label_names = ["chirp_mass", "total_mass", "chi_eff"]

print("Label names:", label_names)
print("y_mean:", y_mean)
print("y_std:", y_std)

assert y_mean.shape == (3,)
assert y_std.shape == (3,)
assert np.all(np.isfinite(y_mean))
assert np.all(np.isfinite(y_std))
assert np.all(y_std > 0)

print("Label statistics validated.")

Label names: ['chirp_mass', 'total_mass', 'chi_eff']
y_mean: [3.74526215e+01 9.49739685e+01 1.02269521e-03]
y_std: [16.48472214 34.65016556  0.44085518]
Label statistics validated.


In [45]:
def inverse_standardize(y_std_space):
    """
    Convert standardized labels/predictions to physical units:
    [chirp_mass, total_mass, chi_eff].
    """
    y_std_space = np.asarray(y_std_space, dtype=np.float64)
    return y_std_space * y_std + y_mean


def standardize(y_phys):
    """
    Convert physical labels to standardized training space.
    """
    y_phys = np.asarray(y_phys, dtype=np.float64)
    return (y_phys - y_mean) / y_std

test_std = np.zeros((1, 3))
test_phys = inverse_standardize(test_std)

print("Zero standardized corresponds to physical mean:")
for name, value in zip(label_names, test_phys[0]):
    print(f"{name}: {value:.6g}")

Zero standardized corresponds to physical mean:
chirp_mass: 37.4526
total_mass: 94.974
chi_eff: 0.0010227


## 7. Función de inferencia

Hace las predicciones usando el modelo entrenado 

In [46]:
def predict_m08(model, X, device="cpu"):
    """
    Run M08 inference.

    Parameters
    ----------
    model : torch.nn.Module
        Loaded M08 model.
    X : np.ndarray
        Shape (n_events, 3, 16384) or (3, 16384).

    Returns
    -------
    pred_std : np.ndarray
        Standardized predictions, shape (n_events, 3).
    pred_phys : np.ndarray
        Physical predictions, shape (n_events, 3).
    emb : np.ndarray
        Embeddings, shape (n_events, 64).
    """
    X = np.asarray(X, dtype=np.float32)

    if X.ndim == 2:
        X = X[None, :, :]

    if X.shape[1:] != (3, 16384):
        raise ValueError(f"Expected X shape (N, 3, 16384), got {X.shape}")

    model = model.to(device)
    model.eval()

    x_tensor = torch.from_numpy(X).to(device)

    with torch.no_grad():
        pred_std_t, emb_t = model(x_tensor, return_embedding=True)

    pred_std = pred_std_t.cpu().numpy()
    emb = emb_t.cpu().numpy()
    pred_phys = inverse_standardize(pred_std)

    return pred_std, pred_phys, emb

In [47]:
X_dummy = np.zeros((1, 3, 16384), dtype=np.float32)

pred_std_dummy, pred_phys_dummy, emb_dummy = predict_m08(model, X_dummy)

print("pred_std_dummy:", pred_std_dummy)
print("pred_phys_dummy:", pred_phys_dummy)
print("emb_dummy shape:", emb_dummy.shape)

pred_std_dummy: [[-2.0862827 -2.5918431 -0.5304327]]
pred_phys_dummy: [[ 3.06083035  5.16617501 -0.23282131]]
emb_dummy shape: (1, 64)


## 8. Load embeddings and preds

In [49]:
# ------------------------------------------------------------
# Load M08 prediction/embedding file
# ------------------------------------------------------------

DATASET_ID = train_cfg["dataset"]["dataset_id"]

prediction_candidates = sorted(
    RESULTS_DIR.rglob(
        f"{DATASET_ID}_SimpleCNN_ResidualDilated*predictions_embeddings*.npz"
    )
)

print("Prediction/embedding candidates:")
for p in prediction_candidates:
    print(" ", p)

print("Number of candidates:", len(prediction_candidates))

assert len(prediction_candidates) >= 1, "No M08 prediction/embedding file found."

M08_PRED_PATH = prediction_candidates[0]
print("Selected:", M08_PRED_PATH)

Prediction/embedding candidates:
  /home/victor/gw/cbc_pe/data/results/bbh_processed_4s_seobnrv4opt_snr10-25_n500_000/bbh_processed_4s_seobnrv4opt_snr10-25_n500_000_SimpleCNN_ResidualDilated_M08_resdilated_emb64_d124_val_cal_test_predictions_embeddings.npz
Number of candidates: 1
Selected: /home/victor/gw/cbc_pe/data/results/bbh_processed_4s_seobnrv4opt_snr10-25_n500_000/bbh_processed_4s_seobnrv4opt_snr10-25_n500_000_SimpleCNN_ResidualDilated_M08_resdilated_emb64_d124_val_cal_test_predictions_embeddings.npz


In [50]:
m08_data = np.load(M08_PRED_PATH, allow_pickle=True)

print("Available keys:")
for key in sorted(m08_data.files):
    value = m08_data[key]
    print(f"{key:25s} shape={value.shape} dtype={value.dtype}")

Available keys:
available_splits          shape=(3,) dtype=<U4
checkpoint_file           shape=() dtype=<U166
dataset_path              shape=() dtype=<U86
emb_cal                   shape=(30000, 64) dtype=float32
emb_test                  shape=(30000, 64) dtype=float32
emb_val                   shape=(40000, 64) dtype=float32
idx_cal                   shape=(30000,) dtype=int64
idx_test                  shape=(30000,) dtype=int64
idx_val                   shape=(40000,) dtype=int64
label_names               shape=(3,) dtype=<U10
label_stats_path          shape=() dtype=<U158
model_config              shape=() dtype=object
pred_cal                  shape=(30000, 3) dtype=float32
pred_test                 shape=(30000, 3) dtype=float32
pred_val                  shape=(40000, 3) dtype=float32
split_path                shape=() dtype=<U142
y_cal                     shape=(30000, 3) dtype=float32
y_mean                    shape=(3,) dtype=float32
y_std                     shape=(3,) dtype

In [51]:
SPLITS = {}

for split in ["val", "cal", "test"]:
    required = [f"pred_{split}", f"y_{split}", f"emb_{split}"]

    missing = [key for key in required if key not in m08_data.files]
    if missing:
        print(f"Skipping split={split}, missing keys:", missing)
        continue

    SPLITS[split] = {
        "pred": np.asarray(m08_data[f"pred_{split}"], dtype=np.float64),
        "y": np.asarray(m08_data[f"y_{split}"], dtype=np.float64),
        "emb": np.asarray(m08_data[f"emb_{split}"], dtype=np.float64),
    }

    idx_key = f"idx_{split}"
    if idx_key in m08_data.files:
        SPLITS[split]["idx"] = np.asarray(m08_data[idx_key])

for split, data in SPLITS.items():
    print(f"\nSplit: {split}")
    for key, value in data.items():
        print(f"  {key:5s}: {value.shape}")


Split: val
  pred : (40000, 3)
  y    : (40000, 3)
  emb  : (40000, 64)
  idx  : (40000,)

Split: cal
  pred : (30000, 3)
  y    : (30000, 3)
  emb  : (30000, 64)
  idx  : (30000,)

Split: test
  pred : (30000, 3)
  y    : (30000, 3)
  emb  : (30000, 64)
  idx  : (30000,)


## 9. Load Mondrian selected configs

In [52]:
# ------------------------------------------------------------
# 9. Load selected Mondrian configurations
# ------------------------------------------------------------

MONDRIAN_DIR = RESULTS_DIR / "mondrian_M08_final_baseline"

print("MONDRIAN_DIR:", MONDRIAN_DIR)
print("Exists:", MONDRIAN_DIR.exists())

for p in sorted(MONDRIAN_DIR.glob("*")):
    print(p.name)

MONDRIAN_DIR: /home/victor/gw/cbc_pe/data/results/bbh_processed_4s_seobnrv4opt_snr10-25_n500_000/mondrian_M08_final_baseline
Exists: False


In [53]:
import pandas as pd

selected_config_path = MONDRIAN_DIR / "selected_configurations.csv"
selected_systems_path = MONDRIAN_DIR / "selected_systems_summary.csv"

selection_df = pd.read_csv(selected_config_path)
selected_systems_df = pd.read_csv(selected_systems_path)

display(selection_df)
display(selected_systems_df)

FileNotFoundError: [Errno 2] No such file or directory: '/home/victor/gw/cbc_pe/data/results/bbh_processed_4s_seobnrv4opt_snr10-25_n500_000/mondrian_M08_final_baseline/selected_configurations.csv'

In [ ]:
MONDRIAN_POLICY = "conservative"

selected_policy_df = selected_systems_df[
    selected_systems_df["final_policy"] == MONDRIAN_POLICY
].copy()

display(selected_policy_df)

In [ ]:
assert set(selected_policy_df["label"]) == set(label_names)
assert len(selected_policy_df) == 3

print("Selected Mondrian policy:", MONDRIAN_POLICY)